# 02. 장기 메모리(Long-Term Memory)

> 사용자 프로필이나 학습된 선호도는 세션을 넘어 살아남아야 해요. `Store` API + `user_id` 네임스페이스로 대화 너머의 영구 메모리를 구축해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **Store API**(`put`/`get`/`search`)로 세션을 넘어 지속되는 장기 메모리를 읽고 쓸 수 있어요
2. `ToolRuntime[Context]`를 사용해 도구에서 Store와 `user_id`에 안전하게 접근할 수 있어요
3. 튜플 기반 **계층적 네임스페이스**로 사용자 데이터를 체계적으로 분류할 수 있어요
4. **동적 프롬프트** 패턴으로 Store 데이터를 시스템 프롬프트에 반영해 개인화된 에이전트를 만들 수 있어요
5. `PostgresSaver` + `PostgresStore`로 프로덕션 수준의 영구 메모리 시스템을 구축할 수 있어요

## 사전 지식

- `01-Short-Term-Memory.ipynb`: MemorySaver, thread_id, checkpointer 개념
- Part 05의 Tool 바인딩 패턴 (create_agent, ToolRuntime)

## 단기 메모리 vs 장기 메모리

앞의 노트북에서 배운 `MemorySaver`(체크포인터)는 **단기 메모리**예요. 같은 `thread_id` 안에서만 대화를 기억하고, 새 thread를 시작하면 기억이 초기화돼요.

**장기 메모리**는 `thread_id`가 달라져도, 심지어 서버가 재시작돼도 사용자 정보를 기억해요. LangGraph는 이를 위해 **Store API**를 제공해요.

> 💡 **핵심 정리**: 단기 메모리와 장기 메모리의 차이를 **웹 서비스**로 비유하면 이해하기 쉬워요. 단기 메모리는 **브라우저 탭(세션)**이에요 — 탭을 닫으면 대화 내용이 사라져요. 장기 메모리는 **회원 데이터베이스(계정)**예요 — 어떤 탭에서 로그인하든 사용자 정보(이름, 설정 등)가 유지돼요.

| 특성 | 단기 메모리 (Checkpointer) | 장기 메모리 (Store) |
|------|--------------------------|--------------------|
| **범위** | 단일 `thread_id` 내 | 여러 세션/thread 공유 |
| **저장 위치** | Checkpointer (상태 스냅샷) | Store (Key-Value DB) |
| **데이터 유형** | 메시지, 임시 상태 | 사용자 프로필, 선호도 |
| **수명** | thread 종료 시 접근 불가 | 명시적 삭제 전까지 유지 |
| **비유** | 브라우저 탭 (세션) | 회원 데이터베이스 (계정) |
| **예시** | 이번 대화 내용 | 사용자 이름, 언어 설정 |

> 🔑 **핵심 개념**: Store는 파일 시스템과 유사한 구조를 가져요. 각 항목은 **(Namespace, Key, Value)** 세 가지로 구성돼요. Namespace는 폴더처럼 데이터를 구분하고, Key는 파일 이름, Value는 JSON 딕셔너리예요.

```{mermaid}
flowchart TD
    subgraph 단기["단기 메모리 (Checkpointer)"]
        T1[thread_id: session_A] --> S1[State: 메시지 3개]
        T2[thread_id: session_B] --> S2[State: 메시지 7개]
    end

    subgraph 장기["장기 메모리 (Store)"]
        NS1["namespace: ('users', 'user_01', 'profile')"] --> K1[key: basic → 이름, 언어]
        NS2["namespace: ('users', 'user_01', 'prefs')"] --> K2[key: style → 대화 스타일]
        NS3["namespace: ('users', 'user_01', 'learned')"] --> K3[key: facts → 학습 데이터]
    end

    T1 -.->|user_id 공유| NS1
    T2 -.->|user_id 공유| NS1

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    class T1,T2 input
    class S1,S2 process
    class NS1,NS2,NS3,K1,K2,K3 storage
```

## 메모리 시스템 전체 아키텍처

단기 메모리와 장기 메모리가 하나의 에이전트 시스템에서 어떻게 함께 동작하는지 살펴볼게요.

```mermaid
flowchart TD
    User([사용자 요청]) --> Agent[에이전트]

    subgraph STM["단기 메모리 (Checkpointer)"]
        CP[MemorySaver / PostgresSaver]
        TH1["thread_001<br>메시지 5개"]
        TH2["thread_002<br>메시지 3개"]
        CP --> TH1
        CP --> TH2
    end

    subgraph LTM["장기 메모리 (Store)"]
        ST[InMemoryStore / PostgresStore]
        NS1["('users', 'u01', 'profile')<br>key: basic"]
        NS2["('users', 'u01', 'prefs')<br>key: style"]
        NS3["('users', 'u02', 'profile')<br>key: basic"]
        ST --> NS1
        ST --> NS2
        ST --> NS3
    end

    Agent -->|"thread_id 기반"| STM
    Agent -->|"user_id 기반"| LTM
    Agent --> Response([응답 반환])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    class User input
    class Agent process
    class CP,ST,TH1,TH2,NS1,NS2,NS3 storage
    class Response output
```

> 💡 **핵심 정리**: 에이전트는 **두 가지 메모리 시스템**에 동시에 접근해요. `thread_id`로 현재 대화를 이어가고, `user_id`로 사용자의 영구 정보를 불러와요. 웹 서비스에서 세션(단기)과 데이터베이스(장기)의 관계와 같아요.

## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY 등을 읽어와요
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택사항)
# ---------------------------------------------------
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-07-Long-Term-Memory"

# LangSmith 추적 설정 완료!

---

## 1. Store API 기본: put / get / search

Store의 기본 사용법부터 살펴볼게요. `InMemoryStore`는 개발/테스트용이에요. 프로덕션에서는 `PostgresStore`를 사용해요.

> 🔑 **핵심 개념**: Store의 세 가지 핵심 연산이에요:
> - `put(namespace, key, value)`: 데이터 저장 (덮어쓰기 지원)
> - `get(namespace, key)`: 특정 키로 데이터 조회
> - `search(namespace, filter=...)`: 네임스페이스 내 필터 검색

> ⚠️ **자주 하는 실수**: `get()`은 항목이 없으면 `None`을 반환해요. `.value`에 접근하기 전에 반드시 `None` 체크를 해야 해요!

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

user_id = "user_123"

namespace = (user_id, "preferences")

store.put(
    namespace,
    "language_settings",
    {
        "preferred_language": "Korean",
        "communication_style": "formal"
    }
)





---

## 2. 도구에서 Store 접근: ToolRuntime[Context]

에이전트의 도구는 `ToolRuntime` 매개변수를 통해 Store에 액세스할 수 있어요. `ToolRuntime`은 LLM에는 숨겨지지만, 실행 시 자동으로 주입돼요.

핵심 패턴:
- `create_agent(store=store, context_schema=Context)`: Store와 Context를 에이전트에 연결
- `runtime.store`: 도구에서 Store 접근
- `runtime.context.user_id`: 현재 사용자 ID 접근

> 💡 **핵심 정리**: `user_id`를 context로 전달하면 한 에이전트로 여러 사용자를 안전하게 처리할 수 있어요. 각 사용자 데이터는 네임스페이스로 완전히 분리되니까요.

> 💡 **실무 팁**: `ToolRuntime`의 타입 힌트 `ToolRuntime[Context]`는 LLM이 이 파라미터를 무시하도록 해요. LLM은 이 파라미터를 모르고 도구를 호출하지만, 런타임에 자동으로 채워져요.

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델: gpt-4o-mini (비용 효율)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 실행: context로 user_id 전달
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 3. Store에 쓰기: 사용자 정보 저장

읽기와 마찬가지로 도구에서 `runtime.store.put()`을 호출해 장기 메모리에 데이터를 저장할 수 있어요. 기존 데이터가 있으면 병합하는 패턴이 실무에서 자주 사용돼요.

> 🔑 **핵심 개념**: 동일한 `(namespace, key)` 조합으로 `put()`을 호출하면 기존 데이터가 **덮어쓰기**돼요. 기존 데이터를 보존하려면 먼저 `get()`으로 불러온 뒤 병합(`{**old, **new}`)해야 해요.

> ⚠️ **자주 하는 실수**: `put()` 전에 `get()`을 하지 않고 바로 덮어쓰면, 이전에 저장된 필드가 전부 사라질 수 있어요. 예를 들어 이름만 업데이트하려다가 이메일까지 삭제되는 실수가 흔해요. 반드시 **get → merge → put** 패턴을 사용하세요.

In [ ]:
from typing_extensions import TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 네임스페이스 계층 구조 시각화

Store의 네임스페이스는 파일 시스템처럼 계층 구조로 데이터를 정리해요.

```mermaid
flowchart TD
    Root["Store (Root)"] --> Users["users/"]
    Users --> U1["user_001/"]
    Users --> U2["user_002/"]

    U1 --> P1["profile/<br>key: basic<br>이름, 나이, 직업"]
    U1 --> S1["settings/<br>key: preferences<br>테마, 알림, 언어"]
    U1 --> A1["activity/<br>key: recent<br>마지막 로그인, 쿼리 수"]

    U2 --> P2["profile/<br>key: basic<br>이름, 나이, 직업"]
    U2 --> S2["settings/<br>key: preferences<br>테마, 알림, 언어"]

    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    class Root storage
    class Users,U1,U2 process
    class P1,S1,A1,P2,S2 input
```

**네임스페이스 튜플과 파일 경로 대응:**

| 네임스페이스 튜플 | 파일 경로 비유 | 용도 |
|------------------|--------------|------|
| `("users", "user_001", "profile")` | `users/user_001/profile/` | 기본 프로필 |
| `("users", "user_001", "settings")` | `users/user_001/settings/` | UI/UX 설정 |
| `("users", "user_001", "activity")` | `users/user_001/activity/` | 활동 기록 |

> 🔑 **핵심 개념**: 네임스페이스 설계는 Store 성능의 핵심이에요. `search()` 할 때 특정 카테고리만 빠르게 조회할 수 있도록, 사용자별/카테고리별로 분리하세요.

---

## 4. 계층적 네임스페이스로 데이터 구조화

네임스페이스는 튜플로 계층 구조를 표현할 수 있어요. `("users", user_id, "profile")`은 파일 시스템의 `users/user_id/profile` 경로와 유사해요.

같은 사용자의 **프로필**, **설정**, **활동 기록**을 논리적으로 분리해서 관리할 수 있어요.

> 💡 **실무 팁**: 네임스페이스 설계가 Store 성능과 유지보수성에 큰 영향을 줘요. 처음부터 계층 구조를 잘 설계하면 나중에 특정 카테고리만 빠르게 검색할 수 있어요.

> ⚠️ **자주 하는 실수**: 모든 데이터를 `("all_data",)` 같은 단일 네임스페이스에 저장하면 데이터가 많아질수록 검색이 느려지고 관리가 어려워요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 5. 동적 프롬프트로 개인화된 에이전트 만들기

`@dynamic_prompt` 데코레이터를 사용하면 Store에 저장된 사용자 선호도를 기반으로 시스템 프롬프트를 **매 요청마다 동적으로 생성**할 수 있어요.

이 패턴은 사용자마다 다른 응답 스타일, 언어, 응답 길이를 적용해야 할 때 매우 유용해요.

> 💡 **핵심 정리**: `@dynamic_prompt`는 Store + 시스템 프롬프트를 연결하는 핵심 패턴이에요. 사용자가 선호도를 업데이트하면 다음 요청부터 바로 반영돼요. 별도의 재배포 없이 에이전트 동작을 실시간으로 커스터마이징할 수 있어요.

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 프롬프트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 사용자별 응답 차이 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 6. 학습하는 에이전트: 카테고리별 장기 기억

사용자와의 대화에서 자동으로 사실을 학습하고, 나중에 회상할 수 있는 에이전트를 만들어볼게요. 카테고리(`personal`, `work`, `hobbies` 등)별로 계층적 네임스페이스를 활용해요.

> 💡 **핵심 정리**: 이 패턴의 핵심은 **학습(`learn_from_interaction`)** 과 **회상(`recall_learned_info`)** 을 도구로 분리한 것이에요. LLM이 스스로 언제 저장하고 언제 조회할지 결정해요. 에이전트에게 '판단'을 위임하는 좋은 예시예요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 세션 간 메모리 공유 흐름

같은 `user_id`로 서로 다른 `thread_id`(세션)에서 접근했을 때 메모리가 어떻게 공유되는지 살펴볼게요.

```mermaid
sequenceDiagram
    participant A as Session A<br/>(thread_001)
    participant Store as Store<br/>(user_id: alice)
    participant B as Session B<br/>(thread_999)

    A->>Store: put("memories", "alice")<br/>이름: Alice
    Note over Store: {note: 이름은 alice}
    B->>Store: search("memories", "alice")
    Store-->>B: {note: 이름은 alice}
    Note over B: 새 세션에서도<br/>Alice를 기억!

    rect rgb(255, 243, 205)
        Note over A,B: 단기 메모리(Checkpointer)는 thread별 독립<br/>장기 메모리(Store)는 user_id별 공유
    end
```

> 💡 **실무 팁**: 이 패턴은 "새 브라우저 탭에서 로그인해도 사용자 설정이 그대로인 것"과 같은 원리예요. `thread_id`가 달라도(새 탭) `user_id`가 같으면(같은 계정) 장기 메모리를 공유해요.

---

## 7. 세션 간 메모리 공유: Cross-Session 검증

장기 메모리의 핵심 특성을 검증해볼게요. 같은 `user_id`를 사용하면 `thread_id`가 달라도 (서로 다른 대화 세션이어도) 동일한 Store 데이터를 공유해요.

> 💡 **핵심 정리**: 단기 메모리(`thread_id` 기반)와 장기 메모리(`user_id` 기반)의 차이를 직접 눈으로 확인하는 중요한 실습이에요. "새 탭에서 로그인해도 사용자 설정이 유지되는 것"과 같은 원리예요.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
from typing import Any
import uuid

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → chatbot → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 8. 프로덕션 메모리: PostgresSaver + PostgresStore

`InMemoryStore`는 프로세스가 종료되면 모든 데이터가 사라져요. 실제 서비스에서는 **영구 저장소**가 필요해요.

LangGraph는 `PostgresSaver`(단기 메모리)와 `PostgresStore`(장기 메모리)를 함께 제공해요:

| 구성 요소 | 역할 | 수명 |
|----------|------|------|
| `PostgresSaver` | thread_id 기반 대화 이력 | 설정에 따라 영구 |
| `PostgresStore` | user_id 기반 장기 메모리 | 명시적 삭제 전까지 |

> ⚠️ **자주 하는 실수**: `PostgresSaver`와 `PostgresStore`는 반드시 **`with` 블록**으로 사용해야 해요. 그렇지 않으면 데이터베이스 연결이 제대로 닫히지 않아 연결 누수(connection leak)가 발생해요.

### PostgreSQL 설정

```bash
# Docker로 PostgreSQL 실행
docker run --name langgraph_ltm \
    -e POSTGRES_PASSWORD=postgres \
    -p 5659:5432 \
    -d postgres:15

# 필요 패키지 설치
pip install langgraph-checkpoint-postgres
```

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 9. LLM 기반 자동 메모리 추출

지금까지는 사용자가 명시적으로 'remember' 키워드를 써야 했어요. 실무에서는 **LLM이 자동으로 중요한 정보를 감지하고 저장**하는 것이 더 자연스러워요.

`create_memory_extractor`는 LLM을 사용해서 대화 메시지에서 구조화된 기억 정보를 자동 추출해요. (외부 패키지 없이 직접 구현해요.)

> 💡 **핵심 정리**: 자동 메모리 추출은 마치 **비서**가 회의 중에 자동으로 중요한 사항을 메모하는 것과 같아요. 사용자가 "제 이름은 민수예요"라고 말하면, 비서가 알아서 "이름: 민수"를 수첩에 적어두는 거죠. 사용자는 "기억해"라고 따로 말할 필요가 없어요.

> 💡 **실무 팁**: 메모리 추출기를 별도 LLM(더 작고 빠른 모델)으로 구성하면 비용을 절약할 수 있어요. 추출은 단순한 분류 작업이라서 작은 모델로도 충분해요.

> ⚠️ **자주 하는 실수**: 모든 대화에서 메모리를 추출하면 불필요한 LLM 호출이 많아져요. 일정 메시지 수 이상이거나 특정 토픽이 감지될 때만 추출을 트리거하는 것이 좋아요.

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from typing import Optional

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 메시지에서 자동 추출
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → chatbot → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 10. 실습: 나만의 개인화 에이전트 만들기

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 학습 이력 기반 개인화 추천 에이전트 구현
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Store API**: `put(namespace, key, value)` / `get(namespace, key)` / `search(namespace, filter=...)` 세 가지 연산으로 장기 메모리를 관리해요
- **Namespace**: 튜플 형태의 계층적 경로로 데이터를 논리적으로 분류해요. `("users", user_id, "profile")`처럼 파일 경로 구조로 설계해요
- **ToolRuntime[Context]**: 도구에서 `runtime.store`와 `runtime.context.user_id`로 Store와 사용자 식별자에 접근해요
- **동적 프롬프트**: `@dynamic_prompt`로 Store 데이터를 시스템 프롬프트에 반영해 사용자별 개인화된 응답을 생성해요
- **Cross-Session 공유**: 같은 `user_id`면 `thread_id`가 달라도 장기 메모리를 공유해요. 반대로 다른 `user_id`면 메모리가 완전히 격리돼요
- **PostgresSaver + PostgresStore**: 프로덕션 환경에서 단기 메모리와 장기 메모리를 각각 영구 저장해요. `with` 블록으로 반드시 연결을 관리해야 해요
- **자동 메모리 추출**: LLM을 사용해 대화에서 중요한 사실을 자동으로 감지하고 Store에 저장할 수 있어요

### 단기 메모리 vs 장기 메모리 최종 정리

| 비교 항목 | 단기 메모리 | 장기 메모리 |
|----------|------------|------------|
| 구현 | `MemorySaver` / `PostgresSaver` | `InMemoryStore` / `PostgresStore` |
| 식별 기준 | `thread_id` | `user_id` (namespace) |
| 데이터 형식 | 메시지 리스트 (State) | JSON Key-Value |
| 조회 방법 | `get_state(config)` | `store.get(namespace, key)` |
| 공유 범위 | 같은 thread 내 | 같은 user_id 전체 |

## 다음 노트북 예고

다음 `Part 08`의 `01-Graph-Based-RAG.ipynb`에서는 **RAG(Retrieval-Augmented Generation) 파이프라인을 LangGraph로 구성**하는 방법을 배워요. 지금까지 배운 단일 에이전트에 기억과 도구를 붙이는 것에서 한 단계 더 나아가, **외부 지식 베이스를 검색해 응답에 활용**하는 도메인 지식 공급 패턴을 살펴볼게요.

학습자 여러분은 이미 v0 기반의 RAG 시스템을 만들어본 경험이 있을 거예요. Part 08에서는 그 경험 위에 **LangGraph의 상태 관리와 조건부 라우팅**을 얹어서 Naive RAG → Agentic → CRAG/Self-RAG → Adaptive RAG로 진화시켜요. 이렇게 다진 RAG 도구 위에서 Part 09 멀티에이전트가 자연스럽게 이어져요.